# Notebook 5 — The Dependency DAG

Blueprint's influence system lets one column depend on another — and those dependencies can chain: `sqft → price → tax`. When you call `emit()`, Blueprint must apply influences in the right order: `price` has to be computed before `tax` can use it.

Blueprint resolves this with a **directed acyclic graph (DAG)**. Every feature is a node; every `Influence(source).on(target)` edge is a directed edge. Before influences are applied, Blueprint runs a **topological sort** on that graph to find a valid evaluation order.

This notebook covers:
- Why evaluation order matters — and what goes wrong without it
- How Blueprint builds the dependency graph from your influences
- Topological sort (Kahn's algorithm) — the math in plain English
- Cycle detection and `BlueprintCycleError`
- Inspecting the DAG with `bp.validate()` and `bp.describe()`
- Visualizing the graph with `networkx`

In [1]:
from blueprint import Blueprint, Feature, Class, Influence
from blueprint.core.dag import DAG, BlueprintCycleError

import pandas as pd
import numpy as np

## Why evaluation order matters

Consider a three-hop chain:

```
units_sold  →  revenue  →  commission
```

`revenue` depends on `units_sold`, and `commission` depends on `revenue`. If Blueprint tried to compute `commission` before `revenue` was ready, it would read 0 (the unmodified derived-feature baseline) and produce the wrong answer.

Blueprint's topological sort guarantees that `units_sold` is processed first, then `revenue`, then `commission` — so every column is fully updated before any downstream column reads from it.

In [2]:
bp = Blueprint(n=6, seed=42)
bp.add_feature(
    Feature("units_sold", dtype=int,   base=100, std=25, clip=(1, None)),
    Feature("revenue",    dtype=float, base=0,   std=0,  derived=True),
    Feature("commission", dtype=float, base=0,   std=0,  derived=True),
)
bp.add_influence(
    Influence("units_sold").on("revenue",    effect="+49.99 per unit"),
    Influence("revenue").on("commission", effect="+0.08 per unit"),
)

df = bp.emit()
print(df.round(2))
print()
print("revenue   = units_sold \u00d7 49.99:", np.allclose(df.revenue, df.units_sold * 49.99))
print("commission = revenue   \u00d7 0.08:  ", np.allclose(df.commission, df.revenue * 0.08))

   units_sold  revenue  commission
0         108  5398.92      431.91
1          74  3699.26      295.94
2         119  5948.81      475.90
3         124  6198.76      495.90
4          51  2549.49      203.96
5          67  3349.33      267.95

revenue   = units_sold × 49.99: True
commission = revenue   × 0.08:  True


Both assertions hold — `commission` correctly reflects the fully-updated `revenue`, not the zero baseline.

## How Blueprint builds the dependency graph

The `DAG` class (`blueprint.core.dag`) is a lightweight directed graph:

```python
class DAG:
    nodes: list      # feature names, insertion order preserved
    edges: list      # (source, target) tuples
    _adj:  dict      # {source: [targets]} adjacency list
```

When `emit()` or `validate()` needs an evaluation order, `Blueprint._topological_order()` does exactly this:

1. Add every feature name as a node
2. For every `Influence(source).on(target)` edge, call `dag.add_edge(source, target)`
3. Call `dag.topological_sort()` to get the ordered list

You can interact with the `DAG` class directly to understand the graph structure:

In [3]:
dag = DAG()
dag.add_node("units_sold")
dag.add_node("revenue")
dag.add_node("commission")
dag.add_edge("units_sold", "revenue")
dag.add_edge("revenue", "commission")

print("Nodes:", dag.nodes)
print("Edges:", dag.edges)
print()
print("Topological order:", dag.topological_sort())

Nodes: ['units_sold', 'revenue', 'commission']
Edges: [('units_sold', 'revenue'), ('revenue', 'commission')]

Topological order: ['units_sold', 'revenue', 'commission']


## Topological sort — Kahn's algorithm

Blueprint uses **Kahn's algorithm** (BFS-based). The key idea: a node is safe to process as soon as all its *predecessors* have been processed.

**Algorithm steps:**

1. Compute the **in-degree** of every node (number of incoming edges)
2. Seed a queue with all nodes whose in-degree is 0 (no predecessors — safe to emit immediately)
3. Repeatedly pop a node from the queue, append it to the output order, and decrement the in-degree of its successors
4. When a successor's in-degree drops to 0, add it to the queue
5. If the output order contains every node, the sort succeeded; otherwise there is a cycle

The algorithm handles **fan-out** (one source feeds multiple targets) and **fan-in** (multiple sources feed one target) naturally:

In [4]:
# Diamond-shaped graph:
#
#   sqft ──> price ──> tax
#   sqft ──> rent  ──> tax
#   beds ──> price
#
dag = DAG()
dag.add_edge("sqft",  "price")
dag.add_edge("beds",  "price")
dag.add_edge("sqft",  "rent")
dag.add_edge("price", "tax")
dag.add_edge("rent",  "tax")

# In-degree = number of incoming edges
in_deg = {n: 0 for n in dag.nodes}
for src, tgt in dag.edges:
    in_deg[tgt] += 1

print("Nodes:", dag.nodes)
print("Edges:", dag.edges)
print()
print("In-degrees:", in_deg)
print()
print("Topological order:", dag.topological_sort())

Nodes: ['sqft', 'price', 'beds', 'rent', 'tax']
Edges: [('sqft', 'price'), ('beds', 'price'), ('sqft', 'rent'), ('price', 'tax'), ('rent', 'tax')]

In-degrees: {'sqft': 0, 'price': 2, 'beds': 0, 'rent': 1, 'tax': 2}

Topological order: ['sqft', 'beds', 'rent', 'price', 'tax']


Reading the result:
- `sqft` and `beds` start (in-degree 0) — no predecessors
- `rent` becomes ready after `sqft` is processed
- `price` becomes ready only after *both* `sqft` and `beds` are processed (in-degree drops from 2 → 1 → 0)
- `tax` is last — it waits for both `price` and `rent`

Blueprint follows exactly this order when applying influences during `emit()`.

## Inspecting the DAG with `describe()`

`bp.describe()` prints a human-readable summary that includes the resolved evaluation order. This is the quickest way to confirm that Blueprint sees the dependency chain you intended.

In [5]:
bp = Blueprint(n=1000, seed=42)
bp.add_feature(
    Feature("sqft",       dtype=int,   base=2_000, std=400, clip=(700, None)),
    Feature("beds",       dtype=int,   base=3,     std=1,   clip=(1, 8)),
    Feature("price",      dtype=float, base=0,     std=0,   derived=True),
    Feature("rent",       dtype=float, base=0,     std=0,   derived=True),
    Feature("annual_tax", dtype=float, base=0,     std=0,   derived=True),
)
bp.add_influence(
    Influence("sqft").on("price", effect="+110 per unit"),
    Influence("beds").on("price", effect="+12000"),
    Influence("sqft").on("rent",  effect="+0.5 per unit"),
    Influence("price").on("annual_tax", effect="+0.012 per unit"),
    Influence("rent").on("annual_tax",  effect="+0.005 per unit"),
)
bp.describe()

Blueprint(n=1000, seed=42)

Features (5):
  sqft                 <class 'int'> base=2000, std=400, clip=(700, None)
  beds                 <class 'int'> base=3, std=1, clip=(1, 8)
  price                <class 'float'> base=0, std=0
  rent                 <class 'float'> base=0, std=0
  annual_tax           <class 'float'> base=0, std=0

Influences (5):
  sqft → price  +110 per unit
  beds → price  +12000
  sqft → rent  +0.5 per unit
  price → annual_tax  +0.012 per unit
  rent → annual_tax  +0.005 per unit

Evaluation order: ['sqft', 'beds', 'rent', 'price', 'annual_tax']


In [6]:
df = bp.emit()
print(df.head(5).round(0))

   sqft  beds     price    rent  annual_tax
0  2122     3  245420.0  1061.0      2950.0
1  1584     2  186240.0   792.0      2239.0
2  2300     3  265000.0  1150.0      3186.0
3  2376     4  273360.0  1188.0      3286.0
4  1220     3  146200.0   610.0      1757.0


The evaluation order `['sqft', 'beds', 'rent', 'price', 'annual_tax']` matches the diamond graph above: both `sqft` and `beds` must precede `price`; both `price` and `rent` must precede `annual_tax`.

## Cycle detection: `BlueprintCycleError`

Kahn's algorithm detects cycles as a by-product: if the topological sort finishes with fewer nodes than the graph contains, some nodes were never reachable (their in-degree never reached 0), which means they are part of a cycle.

Blueprint raises `BlueprintCycleError` in that case. You can trigger it directly on the `DAG` class or let `validate()` / `emit()` catch it for you.

In [7]:
dag = DAG()
dag.add_edge("A", "B")
dag.add_edge("B", "C")
dag.add_edge("C", "A")   # closes the cycle: A → B → C → A

print("has_cycle():", dag.has_cycle())
print()
try:
    dag.topological_sort()
except BlueprintCycleError as e:
    print(f"BlueprintCycleError: {e}")

has_cycle(): True

BlueprintCycleError: Cycle detected involving: ['A', 'B', 'C']


### Cycle caught via `validate()`

In practice, cycles arise from mistake — for example, adding two influences that point at each other. Call `bp.validate()` after assembling the blueprint to catch this before `emit()`.

In [8]:
bp = Blueprint(n=100, seed=42)
bp.add_feature(
    Feature("a", dtype=float, base=10, std=2),
    Feature("b", dtype=float, base=10, std=2),
)
bp.add_influence(
    Influence("a").on("b", effect="+1"),
    Influence("b").on("a", effect="+1"),   # a → b → a: cycle
)

try:
    bp.validate()
    print("No error — DAG is acyclic")
except BlueprintCycleError as e:
    print(f"BlueprintCycleError caught by validate(): {e}")

BlueprintCycleError caught by validate(): Cycle detected in influence graph


## Inspecting the DAG with `validate()`

`bp.validate()` is Blueprint's pre-flight check. It verifies three things before emitting:

1. **Class conditions reference defined features** — no typos in `when=("col", "==", value)` column names
2. **Class overrides reference defined features** — every `override()` target exists
3. **Influence sources and targets reference defined features** — no dangling edges
4. **The influence graph is acyclic** — no circular dependencies

Call it immediately after assembling your blueprint, before any `emit()` call.

In [9]:
bp = Blueprint(n=500, seed=42)
bp.add_feature(
    Feature("sqft",  dtype=int,   base=2_000, std=400,    clip=(700, None)),
    Feature("beds",  dtype=int,   base=3,     std=1,      clip=(1, 8)),
    Feature("price", dtype=float, base=300_000, std=50_000, clip=(50_000, None)),
)
bp.add_influence(
    Influence("sqft").on("price", effect="+110 per unit"),
    Influence("beds").on("price", effect="+12000"),
)

bp.validate()
print("validate() passed — no cycles, all feature names resolve")

validate() passed — no cycles, all feature names resolve


### `validate()` catches undefined feature references

A `ValueError` is raised if an influence targets a column that was never added to the blueprint — catching typos before they silently produce wrong data.

In [10]:
bp = Blueprint(n=100, seed=42)
bp.add_feature(
    Feature("sqft", dtype=int, base=2000, std=400),
    # 'price' is never added
)
bp.add_influence(
    Influence("sqft").on("price", effect="+110 per unit"),
)

try:
    bp.validate()
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: Influence: target 'price' is not a defined feature


## Visualizing the dependency graph

For complex blueprints, a visual graph helps confirm the structure at a glance. The `DAG` object exposes `nodes` and `edges` directly, so it plugs straight into `networkx`.

```bash
pip install networkx matplotlib
```

In [11]:
try:
    import networkx as nx
    import matplotlib.pyplot as plt
    _has_nx = True
except ImportError:
    _has_nx = False
    print("networkx / matplotlib not installed — skipping visualization")
    print("Install with: pip install networkx matplotlib")

In [12]:
if _has_nx:
    # Build the same diamond blueprint
    bp = Blueprint(n=1000, seed=42)
    bp.add_feature(
        Feature("sqft",       dtype=int,   base=2_000, std=400, clip=(700, None)),
        Feature("beds",       dtype=int,   base=3,     std=1,   clip=(1, 8)),
        Feature("price",      dtype=float, base=0,     std=0,   derived=True),
        Feature("rent",       dtype=float, base=0,     std=0,   derived=True),
        Feature("annual_tax", dtype=float, base=0,     std=0,   derived=True),
    )
    bp.add_influence(
        Influence("sqft").on("price", effect="+110 per unit"),
        Influence("beds").on("price", effect="+12000"),
        Influence("sqft").on("rent",  effect="+0.5 per unit"),
        Influence("price").on("annual_tax", effect="+0.012 per unit"),
        Influence("rent").on("annual_tax",  effect="+0.005 per unit"),
    )

    # Extract the DAG from Blueprint internals
    dag = DAG()
    for f in bp.features:
        dag.add_node(f.name)
    for inf in bp._influences:
        for edge in inf.edges:
            dag.add_edge(inf.source, edge["target"])

    # Build networkx graph
    G = nx.DiGraph()
    G.add_nodes_from(dag.nodes)
    G.add_edges_from(dag.edges)

    topo = dag.topological_sort()
    node_color = [topo.index(n) for n in G.nodes]

    pos = nx.spring_layout(G, seed=0)
    fig, ax = plt.subplots(figsize=(7, 4))
    nx.draw_networkx(
        G, pos=pos, ax=ax,
        node_color=node_color, cmap="Blues",
        node_size=1800, font_size=10,
        arrows=True, arrowsize=20,
    )
    ax.set_title("Blueprint dependency DAG (darker = later in evaluation order)")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

The node colour maps to topological position: lighter nodes are evaluated earlier, darker nodes later. This gives an immediate visual read of which columns are roots (no incoming edges) and which are sinks (no outgoing edges).

## Summary

| Concept | What it does |
|---|---|
| `DAG` | Lightweight directed graph — nodes are feature names, edges are influence source→target pairs |
| `dag.topological_sort()` | Kahn's BFS algorithm — returns a valid evaluation order, raises `BlueprintCycleError` if a cycle exists |
| `dag.has_cycle()` | Non-raising cycle check — returns `True` or `False` |
| `BlueprintCycleError` | Raised when the influence graph contains a circular dependency |
| `bp.validate()` | Pre-flight check — verifies feature references and acyclicity before `emit()` |
| `bp.describe()` | Prints the resolved evaluation order alongside the feature/class/influence summary |

**Key rules:**
- Blueprint resolves evaluation order automatically — you never sort influences manually
- Any valid DAG structure is supported: linear chains, fan-out, fan-in, and diamond shapes
- Cycles are always caught — either at `validate()` time or during `emit()`
- Use `derived=True` for columns whose values accumulate entirely from influences (no independent baseline)

**What's next:**
- **Notebook 6** — Blueprint assembly and emission: `validate()`, `describe()`, `emit()`, manifest files, and output formats